# Hyperparameter usage

In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

## Idea

- We need a config file format for storing Hyperparamter combinations on disk.
  - contenders: `json`, `yaml`, `toml`
  - `json`: pros: widespread use, well-defined specification. cons: lacks human readability
  - `yaml`: pros: human readability,  cons: very complicated specification
  - `toml`: pros: human readability, simplicity, cons: not as widely used.
- We need a python class for storing Hyperparamter combinations in memory.
  - contenders: plained nested `dicts`, `attribute-dicts`, `dataclasses`
  - plain dicts: pros: easy to understand ect. cons: annoying to use, no key completion
  - attribute dicst: pros: nice to use, cons: may require extra packages
  - dataclasses: pros: 

### Automatic dataclass

Idea: create a class with signature such that

`cfg = Config(name, x=0, y="abc")` is roughly equivalent to

```python
@dataclass
class name_config:
    x: int
    y: str
    
    @classmethod
    @property
    def __name__(cls):
        return name

cfg = name_config(x=0, y="abc")
```


In [97]:
cf

__main__.name_config

In [178]:
from pydantic import BaseModel

In [181]:
dir(BaseModel(a=0))

['Config',
 '__abstractmethods__',
 '__class__',
 '__class_vars__',
 '__config__',
 '__custom_root_type__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__format__',
 '__ge__',
 '__get_validators__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__json_encoder__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__post_root_validators__',
 '__pre_root_validators__',
 '__pretty__',
 '__private_attributes__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__repr_args__',
 '__repr_name__',
 '__repr_str__',
 '__schema_cache__',
 '__setattr__',
 '__setstate__',
 '__signature__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__validators__',
 '_abc_impl',
 '_calculate_keys',
 '_decompose_class',
 '_enforce_dict_if_root',
 '_get_value',
 '_init_private_attributes',
 '_iter',
 'construct',
 'copy',
 'dict',
 'from_orm',
 'json',
 'parse_file',

In [176]:
from collections.abc import Mapping


@dataclass
class name_config:
    x = 2
    y: str = ""

    def __getitem__(self, key):
        return self.__dict__[key]

    def __setitem__(self, key, value):
        assert key in self.__dict__.keys(), "Cannot add keys!"
        assert isinstance(value, self.__annotations__[key]), "Wrong type!"
        self.__dict__.__setitem__(key, value)

    def __iter__(self):
        return self.__dict__.__iter__()

    def __len__(self):
        return self.__dict__.__len__()


cf = name_config()


dir(cf)

['__annotations__',
 '__class__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setitem__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'x',
 'y']

In [168]:
"z" in cf

False

In [163]:
dict(cf)

KeyError: 0

In [67]:
def Config(, **kwargs)
    class Beta:
        pass

In [69]:
dir(Config())

['Beta',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__']

In [35]:
# The following class would let you do what you want (works in Python 2 & 3):


class Config(dict):
    """Dictionary subclass whose entries can be accessed by attributes (as well
        as normally).

    >>> obj = AttrDict()
    >>> obj['test'] = 'hi'
    >>> print obj.test
    hi
    >>> del obj.test
    >>> obj.test = 'bye'
    >>> print obj['test']
    bye
    >>> print len(obj)
    1
    >>> obj.clear()
    >>> print len(obj)
    0
    """

    __forbidden__ = {
        "clear",
        "copy",
        "fromkeys",
        "get",
        "items",
        "keys",
        "pop",
        "popitem",
        "setdefault",
        "update",
        "values",
    }

    def __init__(self, *args, **kwargs):

        assert not (
            bad_keys := (set(kwargs) & self.__forbidden__)
        ), f"Used forbidden keys {bad_keys}"
        assert not any(
            key.startswith("__") and key.endswith("__") for key in kwargs
        ), f"No dunder keys allowed!"
        super(Config, self).__init__(*args, **kwargs)
        self.__dict__ = self

    # @classmethod
    # def from_nested_dicts(cls, data):
    #     """ Construct nested AttrDicts from nested dictionaries. """
    #     if not isinstance(data, dict):
    #         return data
    #     else:
    #         return cls({key: cls.from_nested_dicts(data[key]) for key in data})

In [45]:
cfg = Config(
    a=1, b=2, optimizer=Config(type="Adam", config=Config(lr=0.001, beta=0.99))
)


cfg

{'a': 1,
 'b': 2,
 'optimizer': {'type': 'Adam', 'config': {'lr': 0.001, 'beta': 0.99}}}

In [22]:
dir(cfg)

['__class__',
 '__class_getitem__',
 '__contains__',
 '__delattr__',
 '__delitem__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__ior__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__or__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__reversed__',
 '__ror__',
 '__setattr__',
 '__setitem__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'a',
 'b',
 'clear',
 'copy',
 'fromkeys',
 'get',
 'items',
 'keys',
 'optimizer',
 'pop',
 'popitem',
 'setdefault',
 'update',
 'values']

In [11]:
from dataclasses import dataclass, asdict


@dataclass
class AutoConf:
    a: int = 0

    @classmethod
    @property
    def __name__(cls):
        return "ja mei"

In [14]:
test()

test(a=0)

In [13]:
dir(test)

['__annotations__',
 '__class__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__name__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'a']